In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sklearn
import numpy as np

## Problem 2.2

**CONTEXT:**

**ReLU (Rectified Linear Unit)**

$$
\text{ReLU}(x)
=
\max(0,x)
$$

**ELU (Exponential Linear Unit)**

$$
\text{ELU}(x)
=
\begin{cases}
x, & x > 0, \\
\alpha(e^x - 1), & x \le 0.
\end{cases}
$$

Typically, $\alpha = 1$.

**GELU (Gaussian Error Linear Unit)**

$$
\text{GELU}(x)
=
x \, \Phi(x)
$$

where $\Phi(x)$ is the cumulative distribution function of the standard normal distribution.

**PROBLEM:** Recall the GELU activation function $g(x)=x\Phi(x)$.

1. Find the derivative $g'(x)$.
2. Evaluate $g'(0)$.
3. Compare the behavior of GELU around $x=0$ with ReLU.

### Solution

**Part 1:**

Find the derivative of $g'(x)$

Note that I will not be taking the derivative of the CDF of the standard normal here.

$$
\begin{align}
    \frac{d}{dx} g(x) &= \frac{d}{dx} x \Phi(x) \\
    &= \Phi(x) + x\Phi'(x)
\end{align}
$$

**Part 2:**

Evaluate $g'(0)$

$$
\begin{align}
    g'(0) &= \Phi(0) + 0\Phi'(0) \\
    &= \Phi(0) \\
    &= 0.5
\end{align}
$$

Explanation: The first derivative of GELU evaluated at 0 results in only the CDF of the standard normal evaluated at 0. That is 0.5 due to the fundamental structure of that random variable.

In [2]:
# Justification:
import scipy.stats
value = scipy.stats.norm.cdf(0, loc=0, scale=1)
print(value)

0.5


**Part 3:**

Compare the behavior of GELU around $x=0$ with ReLU

I'll be keeping this mostly informal. Let's examine ReLU first. It does one of two things. It either provides the value put into it, or 0 for any negative input. This keeps it simple by flattening out negatives, and anything positive is simply itself.

GELU is different as it now incorporates the CDF of the standard normal that scale the input. When we think of that CDF, extreme negative values will have probabilities approaching 0, which scale most negative inputs to 0. However, as the inputs approach 0 the probabilities returned by the CDF grow, resulting in less scaling of the negative inputs and that slight dip shown before 0 on the plot. We see this too with the positive values. They're scaled slightly lower as the CDF continues to climb to 1. Eventually, as the input grows sufficiently beyond 0, the cdf merely becomes a scalar of 1.

The plot makes this tricky to identify, as 1 and 2 seem to match perfectly. But we can see they don't actually quite reach it exactly.

In [3]:
x = torch.linspace(-5, 5, 10 + 1)
yvec = F.gelu(x)

for i, y in enumerate(yvec):
    print(i, x[i], y)

0 tensor(-5.) tensor(-1.4901e-06)
1 tensor(-4.) tensor(-0.0001)
2 tensor(-3.) tensor(-0.0040)
3 tensor(-2.) tensor(-0.0455)
4 tensor(-1.) tensor(-0.1587)
5 tensor(0.) tensor(0.)
6 tensor(1.) tensor(0.8413)
7 tensor(2.) tensor(1.9545)
8 tensor(3.) tensor(2.9960)
9 tensor(4.) tensor(3.9999)
10 tensor(5.) tensor(5.0000)


## Problem 2.3: Consider the example regression MLP with architecture

$$
D \rightarrow 32 \rightarrow 16 \rightarrow 1,
$$

where the two hidden layers use activation functions $g_1$ and $g_2$, and the output layer has no activation function.

1. Write the complete MLP in **matrix-vector form** using weight matrices
   $\mathbf{W}^{(1)}, \mathbf{W}^{(2)}, \mathbf{W}^{(3)}$
   and bias vectors
   $\mathbf{b}^{(1)}, \mathbf{b}^{(2)}, \mathbf{b}^{(3)}$.

2. Specify the dimensions of each weight matrix and bias vector.

3. Now suppose that we remove the nonlinear activation functions, so that

$$
g_1(z)=g_2(z)=z.
$$

Show that the entire network can be written as a linear regression model.


## Solution

Some context:

```
MLP = nn.Sequential(
    nn.Linear(n_features, 32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.ReLU(),

    nn.Linear(16, 1)
)
```

```
for param in MLP.parameters():
    print(param.shape)

torch.Size([32, 8])
torch.Size([32])
torch.Size([16, 32])
torch.Size([16])
torch.Size([1, 16])
torch.Size([1])
```

**Part 1:**

Let's go layer by layer. I'll be referring to the [PyTorch docs](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Linear.html) and using their notation here. They read as follows:

> Applies an affine linear transformation to the incoming data: $y=xA^T+b$.

Of course I'll be using $W^{(i)}$ in place of $A$ here.

$$
\begin{align}
    y^{(1)} &= \textbf{x}\textbf{W}^{(1)T}+\textbf{b}^{(1)} &\text{1st Linear Layer} \\
    h^{(1)} &= g_1(y^{(1)}) &\text{1st activation function} \\
    y^{(2)} &= h^{(1)}\textbf{W}^{(2)T}+\textbf{b}^{(2)} &\text{2nd Linear Layer} \\
    h^{(2)} &= g_2(y^{(2)}) &\text{2nd activation function} \\
    y^{(3)} &= h^{(2)}\textbf{W}^{(3)T}+\textbf{b}^{(3)} &\text{Final Output} \\
\end{align}
$$

**Part 2:**

This part is simple, we can refer to the `parameter.size` output from the context above.

| Object | Dimensions |
|---|---|
| $\mathbf{W}^{(1)}$ | $32\times 8$ |
| $\mathbf{b}^{(1)}$ | $32$ |
| $\mathbf{W}^{(2)}$ | $16\times 32$ |
| $\mathbf{b}^{(2)}$ | $16$ |
| $\mathbf{W}^{(3)}$ | $1\times 16$ |
| $\mathbf{b}^{(3)}$ | $1$ |

**Part 3:**

Removing the activation functions, let's start at the final layer and start substituting backwards.

For this, we can state in advance that $h^{(1)} = y^{(1)}$ and $h^{(2)} = y^{(2)}$ as those are the activation steps being removed.

$$
\begin{align}
    y^{(3)} &= h^{(2)}\textbf{W}^{(3)T}+\textbf{b}^{(3)} & \text{(Start at the end)}\\
    &= y^{(2)}\textbf{W}^{(3)T}+\textbf{b}^{(3)} \\
    &= \left( y^{(1)} \textbf{W}^{(2)T} + b^{(2)} \right) \textbf{W}^{(3)T}+\textbf{b}^{(3)} & \text{(Substitution)} \\
    &= \left( \left( x \textbf{W}^{(1)T} + b^{(1)} \right) \textbf{W}^{(2)T} + b^{(2)} \right) \textbf{W}^{(3)T}+\textbf{b}^{(3)} & \text{(Substitution)} \\
    &= \left( x \textbf{W}^{(1)T}\textbf{W}^{(2)T} + b^{(1)}\textbf{W}^{(2)T} + b^{(2)} \right) \textbf{W}^{(3)T}+\textbf{b}^{(3)} & \text{(Distribution)} \\
    &= x \textbf{W}^{(1)T}\textbf{W}^{(2)T}\textbf{W}^{(3)T} + b^{(1)}\textbf{W}^{(2)T}\textbf{W}^{(3)T} + b^{(2)}\textbf{W}^{(3)T} + b^{(3)} & \text{(Distribution)}
\end{align}
$$

We now have two distinct parts of the right hand side. One that depends on $x$, and another that doesn't. Let's let that second part be our new effective bias term, $b^*$:

$$
b^* = b^{(1)}\textbf{W}^{(2)T}\textbf{W}^{(3)T} + b^{(2)}\textbf{W}^{(3)T} + b^{(3)}
$$

And we'll let the matrix multiplication of the 3 transposed weight matrices be our new effective weight matrix, $W^*$:

$$
W^* = \textbf{W}^{(1)T}\textbf{W}^{(2)T}\textbf{W}^{(3)T}
$$

Substituting those back in gives us:

$$
y^{(3)} = \textbf{x}W^* + b^*
$$

which is in the form of a linear regression, completing the problem.

## Problem 2.4 - Implementing the Gaussian NLL Loss

Throughout this section, we used PyTorch's built-in `nn.GaussianNLLLoss()` to train our probabilistic neural network. In this problem, you will implement the Gaussian negative log-likelihood (NLL) yourself and verify your implementation.

#### Part (a): Implement the loss function

Using our previous derivation of the Gaussian NLL, define your own function

```python
def gaussian_nll(y, mean, var):
    # Your code here
```

that computes the **average Gaussian negative log-likelihood** over a batch of observations.

You may refer to the PyTorch documentation for `nn.GaussianNLLLoss` when implementing your function: https://docs.pytorch.org/docs/2.13/generated/torch.nn.GaussianNLLLoss.html

#### Part (b): Verify your implementation

Use the trained `ProbMLP` model to obtain the predicted means and variances for the **test data**.

Then compute the test loss in two different ways:

1. Using your own `gaussian_nll()` function.
2. Using PyTorch's `nn.GaussianNLLLoss()`.

Print both loss values and compare them. Do the two implementations produce approximately the same test loss?